초기 설정

In [7]:
# 1. 기존에 설치 실패한 폴더 완전 삭제 (충돌 방지)
%cd /content
!rm -rf Marabou

# 2. 필수 빌드 패키지 설치
!apt-get update
!apt-get install -y cmake build-essential python3-dev

# 3. Marabou 공식 리포지토리 클론
!git clone https://github.com/NeuralNetworkVerification/Marabou.git

# 4. 폴더 이동 후 CMake 환경 설정 (호환성 무시 플래그 추가)
%cd /content/Marabou
!mkdir build
%cd build
!cmake .. -DBUILD_PYTHON=ON -DCMAKE_POLICY_VERSION_MINIMUM=3.5

# 5. 실제 컴파일 진행 (멀티코어 사용, 약 3~5분 소요)
!cmake --build . -j 2

# 6. 원래 작업 경로로 복귀
%cd /content

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  152 |                SA( I, J ) = A( I, J )
      |                            1
dlat2s.f:163:28:

  163 |                SA( I, J ) = A( I, J )
      |                            1
dlansf.f:233:40:

  233 |       DOUBLE PRECISION   SCALE, S, VALUE, AA, TEMP
      |                                        ^
dorcsd.f:334:31:

  334 |      $                   ITAUQ2, J, LBBCSDWORK, LBBCSDWORKMIN,
      |                               ^
dorcsd.f:333:69:

  333 |      $                   IORGLQ, IORGQR, IPHI, ITAUP1, ITAUP2, ITAUQ1,
      |                                                                     ^
dorcsd.f:333:61:

  333 |      $                   IORGLQ, IORGQR, IPHI, ITAUP1, ITAUP2, ITAUQ1,
      |                                                             ^
dorcsd.f:333:53:

  333 |      $                   IORGLQ, IORGQR, IPHI, ITAUP1, ITAUP2, ITAUQ1,
      |                                                     ^
dorcsd.f:333:39:

  333 

검증

In [8]:
import sys
import os

# 방금 직접 컴파일한 Marabou 폴더 경로를 시스템에 추가
marabou_path = '/content/Marabou'
if marabou_path not in sys.path:
    sys.path.append(marabou_path)

# 로드 테스트
try:
    from maraboupy import Marabou
    print("✅ Marabou 설치 및 로드 대성공!")
except ImportError as e:
    print(f"❌ 로드 실패: {e}")

❌ 로드 실패: No module named 'maraboupy'


외부 모델 준비

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

# 1. 소형 CNN 정의
class SmallCNN(nn.Module):
    def __init__(self):
        super(SmallCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()
        self.fc = nn.Linear(4 * 28 * 28, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# 2. MNIST 데이터 로드 및 1 에폭 학습
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

model = SmallCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

model.train()
for images, labels in train_loader:
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()


# 3. 모델을 ONNX 포맷으로 변환하여 저장
dummy_input = torch.randn(1, 1, 28, 28)
torch.onnx.export(model, dummy_input, "my_small_model.onnx",
                  export_params=True, opset_version=10,
                  do_constant_folding=True,
                  input_names=['input'], output_names=['output'])
print("✅ ONNX 모델 저장 완료: my_small_model.onnx")

/tmp/ipykernel_1649/38528692.py:41: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(model, dummy_input, "my_small_model.onnx",
W0508 14:09:57.221000 1649 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 10 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0508 14:09:57.758000 1649 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_w

[torch.onnx] Obtain model graph for `SmallCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SmallCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
✅ ONNX 모델 저장 완료: my_small_model.onnx


In [17]:
import os
import sys

# 1. Marabou 빌드 결과물이 실제로 있는 위치로 이동
%cd /content/Marabou/maraboupy

# 2. 빌드된 공유 라이브러리 파일(.so)이 있는지 확인하고 이름을 표준으로 변경
# 파이썬 3.12 환경에서 인식할 수 있도록 이름을 맞춰주는 작업입니다.
!find . -name "*.so" -exec cp {} ./MarabouCore.so \;

# 3. 최상위 경로로 복사 (파이썬이 바로 찾을 수 있게)
%cd /content
!cp -r /content/Marabou/maraboupy /content/maraboupy

# 4. 경로 다시 추가
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

# 5. 임포트 재시도
try:
    from maraboupy import Marabou
    print("✅ [대성공] 드디어 Marabou 로드에 성공했습니다!")
except ImportError as e:
    print(f"❌ 여전히 로드 실패. 에러 메시지: {e}")
    print("💡 팁: '런타임' -> '세션 다시 시작'을 누른 후, '빌드 코드'부터 다시 천천히 실행해 보세요.")

/content/Marabou/maraboupy
cp: './MarabouCore.so' and './MarabouCore.so' are the same file
/content
✅ [대성공] 드디어 Marabou 로드에 성공했습니다!


In [18]:
!pip install onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 57.4 MB/s eta 0:00:00


test 진행

In [22]:
import sys
import os
import numpy as np

# 1. Marabou 경로 설정 및 클래스 로드 해결
marabou_path = '/content'
if marabou_path not in sys.path:
    sys.path.insert(0, marabou_path)

try:
    from maraboupy import Marabou
    from maraboupy.MarabouNetworkONNX import MarabouNetworkONNX
    from maraboupy import MarabouCore

    Marabou.MarabouNetworkONNX = MarabouNetworkONNX
    print("✅ Marabou 및 ONNX 클래스 로드 성공!")
except Exception as e:
    print(f"❌ 로드 실패: {e}")
    sys.exit(1)

# ---------------------------------------------------------------------------
# Problem 2: 외부 모델 검증 (MNIST CNN)
# ---------------------------------------------------------------------------

# 1. 저장된 외부 ONNX 모델 로드
try:
    network = MarabouNetworkONNX("my_small_model.onnx")
    print("✅ 외부 ONNX 모델 로드 완료.")
except Exception as e:
    print(f"❌ 모델 로드 실패: {e}")
    sys.exit(1)

# 2. 입력 및 출력 변수 설정
inputVars = network.inputVars[0]
outputVars = network.outputVars[0]

# 3. 검증 쿼리 구성 (Robustness 검증)
eps = 0.01
reference_image = np.zeros((1, 1, 28, 28))
predicted_class = 3

for i, var in enumerate(inputVars.flatten()):
    val = reference_image.flatten()[i]
    network.setLowerBound(var, max(0.0, val - eps))
    network.setUpperBound(var, min(1.0, val + eps))

target_class = predicted_class
other_class = 4

eq = Marabou.Equation(MarabouCore.Equation.GE)
eq.addAddend(1, outputVars.flatten()[other_class])
eq.addAddend(-1, outputVars.flatten()[target_class])
eq.setScalar(0)
network.addEquation(eq)

print("🚀 검증 엔진 실행 중...")
options = Marabou.createOptions(verbosity=0)
result = network.solve(options=options)

status, vals, stats = result

print("\n" + "="*30)
if status == "sat":
    print("결과: SAT (반례 발견)")
elif status == "unsat":
    print("결과: UNSAT (검증 완료)")
else:
    print(f"결과: {status}")
print("="*30)

# 통계 정보가 있는 경우 간단히 표시
if stats:
    print("검증이 완료되었습니다.")

✅ Marabou 및 ONNX 클래스 로드 성공!
✅ 외부 ONNX 모델 로드 완료.
🚀 검증 엔진 실행 중...
unsat

결과: UNSAT (검증 완료)
검증이 완료되었습니다.
